# 📊 EDA — Analisis Temuan Uang Palsu
## KPw Bank Indonesia Kota Pematang Siantar | 2018–2025
---
Notebook ini melakukan **Exploratory Data Analysis (EDA)** terhadap data temuan uang palsu (`data_up.xlsx`).

**Struktur kolom:**
| Kolom | Tipe | Keterangan |
|---|---|---|
| TAHUN | int | Tahun pencatatan |
| NOMOR SERI 1 | str | Nomor seri uang |
| JUMLAH LEMBAR | int | Jumlah lembar uang |
| TAHUN EMISI | int | Tahun emisi uang |
| PECAHAN | int | Nominal (5rb – 100rb) |
| SATKER | str | Satuan kerja wilayah |
| HASIL KLARIFIKASI | str | Palsu / Asli / Menunggu |
| STATUS | str | Status analisa |

## 1. Setup & Import Library

In [ ]:
# Install library yang dibutuhkan (jalankan sekali)
!pip install -q pandas matplotlib seaborn plotly openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# Styling
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

COLOR_RED    = '#c0392b'
COLOR_NAVY   = '#2c3e50'
COLOR_BLUE   = '#2980b9'
COLOR_GOLD   = '#f39c12'
COLOR_GREEN  = '#27ae60'
PALETTE_CAT  = [COLOR_RED, COLOR_GOLD, COLOR_GREEN, COLOR_BLUE, '#8e44ad', '#16a085']

print("✅ Library berhasil diimport.")

## 2. Load Data

In [ ]:
# --- OPSI A: Upload file manual (Colab) ---
from google.colab import files
uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]
print(f"File diupload: {FILE_PATH}")

In [ ]:
# --- OPSI B: Dari Google Drive (comment opsi A dulu jika pakai ini) ---
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/path/ke/data_up.xlsx'

In [ ]:
# Load data
df = pd.read_excel(FILE_PATH)

# Cleaning awal
df['TAHUN']         = df['TAHUN'].astype(int)
df['TAHUN EMISI']   = df['TAHUN EMISI'].astype(int)
df['PECAHAN']       = df['PECAHAN'].astype(int)
df['JUMLAH LEMBAR'] = df['JUMLAH LEMBAR'].fillna(0).astype(int)
df['NOMOR SERI 1']  = df['NOMOR SERI 1'].astype(str).str.strip()
df['STATUS']        = df['STATUS'].fillna('Tidak Diketahui')

# Kolom turunan
df['NILAI NOMINAL'] = df['PECAHAN'] * df['JUMLAH LEMBAR']
df['LABEL PECAHAN'] = df['PECAHAN'].apply(lambda x: f'Rp {x:,}'.replace(',','.'))

print(f"Data berhasil dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom")
df.head()

## 3. Gambaran Umum Dataset

In [ ]:
# Info kolom & tipe data
df.info()

In [ ]:
# Statistik deskriptif kolom numerik
df[['TAHUN','JUMLAH LEMBAR','TAHUN EMISI','PECAHAN','NILAI NOMINAL']].describe().T

In [ ]:
# Cek missing values
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal baris dengan nilai kosong: {df.isnull().any(axis=1).sum()}")

In [ ]:
# Ringkasan nilai unik per kolom kategorik
for col in ['TAHUN','SATKER','PECAHAN','HASIL KLARIFIKASI','STATUS','TAHUN EMISI']:
    vals = sorted(df[col].unique())
    print(f"\n[{col}] — {len(vals)} nilai unik:")
    print(vals)

## 4. Key Performance Indicators

In [ ]:
total_laporan   = len(df)
total_lembar    = df['JUMLAH LEMBAR'].sum()
total_nilai     = df['NILAI NOMINAL'].sum()
total_seri_unik = df['NOMOR SERI 1'].nunique()
total_satker    = df['SATKER'].nunique()

print("=" * 45)
print(f"  Total Laporan (baris)   : {total_laporan:>10,}")
print(f"  Total Lembar Uang       : {total_lembar:>10,}")
print(f"  Total Nilai Nominal     : Rp {total_nilai:>12,.0f}".replace(',','.'))
print(f"  Nomor Seri Unik         : {total_seri_unik:>10,}")
print(f"  SATKER Terlibat         : {total_satker:>10,}")
print("=" * 45)

## 5. Tren Temuan per Tahun

In [ ]:
tren = df.groupby('TAHUN').agg(
    Total_Laporan   = ('JUMLAH LEMBAR', 'count'),
    Total_Lembar    = ('JUMLAH LEMBAR', 'sum'),
    Total_Nilai     = ('NILAI NOMINAL',  'sum'),
    Seri_Unik       = ('NOMOR SERI 1',  'nunique'),
).reset_index()

print(tren.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Total Lembar
axes[0].plot(tren['TAHUN'], tren['Total_Lembar'],
             marker='o', linewidth=2.5, color=COLOR_RED, markersize=8)
axes[0].fill_between(tren['TAHUN'], tren['Total_Lembar'], alpha=0.12, color=COLOR_RED)
for x, y in zip(tren['TAHUN'], tren['Total_Lembar']):
    axes[0].annotate(str(y), (x, y), textcoords='offset points', xytext=(0,8),
                     ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Total Lembar Uang per Tahun')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Jumlah Lembar')
axes[0].set_xticks(tren['TAHUN'])

# Total Laporan
axes[1].bar(tren['TAHUN'], tren['Total_Laporan'],
            color=COLOR_NAVY, alpha=0.82, width=0.6)
for x, y in zip(tren['TAHUN'], tren['Total_Laporan']):
    axes[1].text(x, y + 2, str(y), ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Jumlah Laporan per Tahun')
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Jumlah Laporan')
axes[1].set_xticks(tren['TAHUN'])

plt.tight_layout()
plt.savefig('tren_tahunan.png', bbox_inches='tight')
plt.show()

## 6. Analisis Hasil Klarifikasi

In [ ]:
klarifikasi = df.groupby('HASIL KLARIFIKASI').agg(
    Jumlah_Laporan = ('HASIL KLARIFIKASI', 'count'),
    Total_Lembar   = ('JUMLAH LEMBAR', 'sum'),
).reset_index().sort_values('Total_Lembar', ascending=False)

klarifikasi['% Lembar'] = (klarifikasi['Total_Lembar'] / klarifikasi['Total_Lembar'].sum() * 100).round(2)
print(klarifikasi.to_string(index=False))

In [ ]:
WARNA_KLA = {
    'Palsu': COLOR_RED,
    'Palsu - Menunggu Klasifikasi': COLOR_GOLD,
    'Asli': COLOR_GREEN,
    'Asli - Rusak': COLOR_BLUE,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
colors = [WARNA_KLA.get(k, '#95a5a6') for k in klarifikasi['HASIL KLARIFIKASI']]
wedges, texts, autotexts = axes[0].pie(
    klarifikasi['Total_Lembar'],
    labels=klarifikasi['HASIL KLARIFIKASI'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    wedgeprops=dict(width=0.58),   # donut
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
axes[0].set_title('Proporsi Hasil Klarifikasi\n(berdasarkan Total Lembar)')

# Bar per tahun & klarifikasi
pivot_kla = df.groupby(['TAHUN','HASIL KLARIFIKASI'])['JUMLAH LEMBAR'].sum().unstack(fill_value=0)
pivot_kla.plot(kind='bar', stacked=True, ax=axes[1],
               color=[WARNA_KLA.get(c,'#95a5a6') for c in pivot_kla.columns],
               width=0.65, edgecolor='white')
axes[1].set_title('Distribusi Klarifikasi per Tahun')
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Jumlah Lembar')
axes[1].set_xticklabels(pivot_kla.index, rotation=0)
axes[1].legend(title='Hasil Klarifikasi', fontsize=8,
               loc='upper left', framealpha=0.7)

plt.tight_layout()
plt.savefig('distribusi_klarifikasi.png', bbox_inches='tight')
plt.show()

## 7. Analisis Pecahan Nominal

In [ ]:
pecahan_df = df.groupby('PECAHAN').agg(
    Jumlah_Laporan = ('JUMLAH LEMBAR', 'count'),
    Total_Lembar   = ('JUMLAH LEMBAR', 'sum'),
    Total_Nilai    = ('NILAI NOMINAL',  'sum'),
).reset_index().sort_values('Total_Lembar', ascending=False)

pecahan_df['Label'] = pecahan_df['PECAHAN'].apply(lambda x: f'Rp {x:,}'.replace(',','.'))
pecahan_df['% Lembar'] = (pecahan_df['Total_Lembar'] / pecahan_df['Total_Lembar'].sum() * 100).round(2)
print(pecahan_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart pecahan
bars = axes[0].bar(pecahan_df['Label'], pecahan_df['Total_Lembar'],
                   color=PALETTE_CAT[:len(pecahan_df)], width=0.55, edgecolor='white')
axes[0].set_title('Total Lembar per Pecahan')
axes[0].set_xlabel('Pecahan')
axes[0].set_ylabel('Jumlah Lembar')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)),
                 ha='center', fontsize=9, fontweight='bold')

# Pecahan per tahun (heatmap)
heat_piv = df.groupby(['TAHUN','PECAHAN'])['JUMLAH LEMBAR'].sum().unstack(fill_value=0)
heat_piv.columns = [f'Rp {c:,}'.replace(',','.') for c in heat_piv.columns]
sns.heatmap(heat_piv, ax=axes[1], cmap='Reds', annot=True, fmt='g',
            linewidths=0.5, cbar_kws={'label':'Jumlah Lembar'})
axes[1].set_title('Heatmap Temuan: Tahun × Pecahan')
axes[1].set_xlabel('Pecahan')
axes[1].set_ylabel('Tahun')

plt.tight_layout()
plt.savefig('analisis_pecahan.png', bbox_inches='tight')
plt.show()

## 8. Analisis Tahun Emisi Uang Palsu

In [ ]:
# Filter hanya uang palsu
palsu_df = df[df['HASIL KLARIFIKASI'].str.contains('Palsu', na=False)].copy()

emisi = palsu_df.groupby('TAHUN EMISI').agg(
    Total_Lembar = ('JUMLAH LEMBAR', 'sum'),
    Jumlah_Laporan = ('JUMLAH LEMBAR', 'count'),
).reset_index().sort_values('Total_Lembar', ascending=False)

print("=== Distribusi Uang Palsu per Tahun Emisi ===")
print(emisi.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart tahun emisi
axes[0].barh(
    emisi['TAHUN EMISI'].astype(str),
    emisi['Total_Lembar'],
    color=COLOR_RED, alpha=0.85
)
axes[0].set_title('Total Lembar Palsu per Tahun Emisi')
axes[0].set_xlabel('Jumlah Lembar')
axes[0].set_ylabel('Tahun Emisi')
for i, v in enumerate(emisi['Total_Lembar']):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9, fontweight='bold')
axes[0].invert_yaxis()

# Stacked bar: tahun emisi per tahun temuan
emisi_pivot = palsu_df.groupby(['TAHUN','TAHUN EMISI'])['JUMLAH LEMBAR'].sum().unstack(fill_value=0)
emisi_pivot.columns = emisi_pivot.columns.astype(str)
emisi_pivot.plot(kind='bar', stacked=True, ax=axes[1],
                 colormap='Set2', width=0.65, edgecolor='white')
axes[1].set_title('Komposisi Tahun Emisi Palsu per Tahun Temuan')
axes[1].set_xlabel('Tahun Temuan')
axes[1].set_ylabel('Jumlah Lembar')
axes[1].set_xticklabels(emisi_pivot.index, rotation=0)
axes[1].legend(title='Tahun Emisi', fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('tahun_emisi.png', bbox_inches='tight')
plt.show()

## 9. Analisis Nomor Seri

In [ ]:
# Top 10 nomor seri berdasarkan total lembar
top10 = (
    df.groupby('NOMOR SERI 1')
    .agg(
        Jumlah_Laporan = ('NOMOR SERI 1', 'count'),
        Total_Lembar   = ('JUMLAH LEMBAR', 'sum'),
    )
    .reset_index()
    .sort_values('Total_Lembar', ascending=False)
    .head(10)
)

print("=== Top 10 Nomor Seri (Total Lembar) ===")
print(top10.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors_top = [COLOR_RED if i == 0 else COLOR_NAVY for i in range(len(top10))]
bars = ax.barh(
    top10['NOMOR SERI 1'][::-1],
    top10['Total_Lembar'][::-1],
    color=colors_top[::-1], height=0.65
)
ax.set_title('Top 10 Nomor Seri — Paling Banyak Ditemukan')
ax.set_xlabel('Total Lembar')
ax.set_ylabel('Nomor Seri')
ax.tick_params(axis='y', labelsize=10)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height()/2,
            str(int(w)), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('top10_nomor_seri.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Nomor Seri JUARA per Tahun ──
# Teknik: groupby TAHUN + NOMOR SERI → rank 1 per tahun
seri_per_tahun = (
    df.groupby(['TAHUN', 'NOMOR SERI 1'])
    .agg(
        Total_Lembar    = ('JUMLAH LEMBAR', 'sum'),
        Jumlah_Laporan  = ('NOMOR SERI 1', 'count'),
    )
    .reset_index()
)

# Rank per tahun (descending)
seri_per_tahun['Rank'] = (
    seri_per_tahun
    .groupby('TAHUN')['Total_Lembar']
    .rank(method='first', ascending=False)
    .astype(int)
)

juara = (
    seri_per_tahun[seri_per_tahun['Rank'] == 1]
    .sort_values('TAHUN')
    .reset_index(drop=True)
)

print("=== 🥇 Nomor Seri JUARA per Tahun ===")
print(juara[['TAHUN','NOMOR SERI 1','Total_Lembar','Jumlah_Laporan']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(juara['TAHUN'], juara['Total_Lembar'],
              color=COLOR_RED, alpha=0.85, width=0.6)
ax.set_title('Nomor Seri Juara per Tahun (Total Lembar Terbanyak)')
ax.set_xlabel('Tahun')
ax.set_ylabel('Total Lembar')
ax.set_xticks(juara['TAHUN'])

for bar, seri in zip(bars, juara['NOMOR SERI 1']):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            seri, ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=COLOR_NAVY,
            fontfamily='monospace')
    ax.text(bar.get_x() + bar.get_width()/2, h/2,
            str(int(h)), ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('juara_per_tahun.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Top 3 Nomor Seri per Tahun ──
top3_per_tahun = (
    seri_per_tahun[seri_per_tahun['Rank'] <= 3]
    .sort_values(['TAHUN', 'Rank'])
    [['TAHUN','Rank','NOMOR SERI 1','Total_Lembar','Jumlah_Laporan']]
    .reset_index(drop=True)
)

# Tampilkan dengan warna per rank
def highlight_rank(row):
    if row['Rank'] == 1:
        return ['background-color: #fdedec; font-weight:bold'] * len(row)
    elif row['Rank'] == 2:
        return ['background-color: #fef9e7'] * len(row)
    else:
        return ['background-color: #eafaf1'] * len(row)

print("=== Top 3 Nomor Seri per Tahun ===")
top3_per_tahun.style.apply(highlight_rank, axis=1)

## 10. Analisis Status Analisa

In [ ]:
status_df = df.groupby('STATUS').agg(
    Total_Laporan = ('STATUS', 'count'),
    Total_Lembar  = ('JUMLAH LEMBAR', 'sum'),
).reset_index().sort_values('Total_Lembar', ascending=False)

status_df['% Laporan'] = (status_df['Total_Laporan'] / status_df['Total_Laporan'].sum() * 100).round(2)
print(status_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
palette = [COLOR_GREEN, COLOR_BLUE, COLOR_GOLD, COLOR_RED]
bars = ax.bar(status_df['STATUS'], status_df['Total_Laporan'],
              color=palette[:len(status_df)], width=0.5)
ax.set_title('Distribusi Status Analisa')
ax.set_xlabel('Status')
ax.set_ylabel('Jumlah Laporan')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            str(int(h)), ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('status_analisa.png', bbox_inches='tight')
plt.show()

## 11. Insight Tambahan

In [ ]:
# Persentase Palsu vs Non-Palsu per Tahun
palsu_pct = (
    df.groupby(['TAHUN','HASIL KLARIFIKASI'])['JUMLAH LEMBAR']
    .sum()
    .unstack(fill_value=0)
    .assign(Total=lambda x: x.sum(axis=1))
)
# Hitung % palsu (gabung kedua kategori palsu)
palsu_cols = [c for c in palsu_pct.columns if 'Palsu' in str(c)]
palsu_pct['% Palsu'] = palsu_pct[palsu_cols].sum(axis=1) / palsu_pct['Total'] * 100
print("=== Persentase Palsu per Tahun ===")
print(palsu_pct[palsu_cols + ['Total','% Palsu']].round(2).to_string())

In [ ]:
# Nomor seri yang muncul lintas 3+ tahun berbeda
seri_multi_tahun = (
    df.groupby('NOMOR SERI 1')['TAHUN']
    .nunique()
    .reset_index(name='Jumlah_Tahun')
    .query('Jumlah_Tahun >= 3')
    .sort_values('Jumlah_Tahun', ascending=False)
    .merge(
        df.groupby('NOMOR SERI 1')['JUMLAH LEMBAR'].sum().reset_index(name='Total_Lembar'),
        on='NOMOR SERI 1'
    )
)
print(f"=== Nomor Seri Muncul di ≥ 3 Tahun Berbeda: {len(seri_multi_tahun)} nomor ===")
print(seri_multi_tahun.head(15).to_string(index=False))

In [ ]:
# Pecahan yang paling sering dipalsukan per tahun
palsu_pecahan = (
    df[df['HASIL KLARIFIKASI'].str.contains('Palsu', na=False)]
    .groupby(['TAHUN','PECAHAN'])['JUMLAH LEMBAR']
    .sum()
    .unstack(fill_value=0)
)
palsu_pecahan.columns = [f'Rp {c:,}'.replace(',','.') for c in palsu_pecahan.columns]
palsu_pecahan['Terbanyak'] = palsu_pecahan.idxmax(axis=1)
print("=== Pecahan Palsu Terbanyak per Tahun ===")
print(palsu_pecahan.to_string())

## 12. Ekspor Data Ringkasan

In [ ]:
import os
os.makedirs('output_eda', exist_ok=True)

# Tren tahunan
tren.to_csv('output_eda/tren_tahunan.csv', index=False)

# Klarifikasi
klarifikasi.to_csv('output_eda/klarifikasi.csv', index=False)

# Top 10 nomor seri
top10.to_csv('output_eda/top10_nomor_seri.csv', index=False)

# Juara per tahun
juara.to_csv('output_eda/juara_per_tahun.csv', index=False)

# Top 3 per tahun
top3_per_tahun.to_csv('output_eda/top3_per_tahun.csv', index=False)

# Seri multi-tahun
seri_multi_tahun.to_csv('output_eda/seri_multi_tahun.csv', index=False)

print("✅ Semua file tersimpan di folder output_eda/")

In [ ]:
# (Opsional) Download semua file sekaligus sebagai ZIP
import shutil
shutil.make_archive('output_eda', 'zip', 'output_eda')
from google.colab import files
files.download('output_eda.zip')

## 13. Ringkasan Temuan EDA

| # | Temuan |
|---|--------|
| 1 | Data mencakup **2.516 laporan** dari 2018–2025, total **2.539 lembar** uang |
| 2 | Dominasi kategori **Palsu - Menunggu Klasifikasi** di tahun-tahun awal |
| 3 | Pecahan **Rp 50.000** paling banyak dipalsukan |
| 4 | Tahun emisi **2016** mendominasi uang palsu yang ditemukan |
| 5 | Nomor seri **REN388132** muncul 47× (49 lembar), tertinggi sepanjang periode |
| 6 | Terdapat nomor seri yang muncul di **3+ tahun berbeda** — indikasi satu lembar beredar lama |
| 7 | Temuan meningkat signifikan di **2024** (873 lembar/keping) |

> 📌 Gunakan `app_uangpalsu.py` (Streamlit) untuk eksplorasi interaktif dashboard dengan filter.